In [1]:
import pdfplumber
import pandas as pd

In [2]:
def table_parsing(pdf_path: str)->list[str]:
    """ 
    Extract text and table clenly from a PDF, ensuring table intact, self-containt text
    block (atomic units) rather than being cut up by naive character chunker
    """
    chunks = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            print(f"Processing page {page_num + 1} ...")

            # 1. Extract table explicitly on the page
            tables = page.extract_tables()
            table_bboxes = [
                t.bbox for t in page.find_tables()
            ]

            # 2. Extract reguler text while ignoring or clernly handling tables area
            # to keep it simple: we extract non-text block and format table as Markdown
            
            # Convert extracted table to Markdown
            page_chunks = []
            for table in tables:
                if not table or len(table) < 2:
                    continue

                # Clean row and set headers
                headers = [h or f"col_{i}" for i, h in enumerate(table[0])]
                df = pd.DataFrame(table[1:], columns=headers)

                # Convert table DataFrame into structured Markdown string
                table_markdown = df.to_markdown(index=False)
                formated_table = (
                    f"\n--- [Table Start: Page {page_num + 1}] ---\n"
                    f"{table_markdown}\n"
                    f"---[Table End] ---\n"
                )
                
                page_chunks.append(formated_table)

                # 3. Extract normal text from the page (excluding table boundary boxes if needed)
                # extract word and group them into text blocks
                text = page.extract_text(layout=False)
                if text:
                    # Standard based character chunking applied ONLY to non-table prose text
                    prose_chunks = simple_text_chunk(text, chunk_size=500, overlap=50)
                    page_chunks.extend(prose_chunks)
                
                chunks.extend(page_chunks) 
            
    return chunks                

In [3]:
def simple_text_chunk(text, chunk_size:int=500, overlap:int=50) -> list[str]:
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap

    return chunks    

In [4]:
import ollama
import psycopg2

In [5]:
# Configuration
DB_CONFIG = {
    "dbname": "company_policy",
    "user": "pgvector",
    "password": "pgvectorpasswd",
    "host": "localhost",
    "port": "5432"
}

In [6]:
# Setup database 
def setup_database():
    """ Ensure pgvector extension and table is exists """
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    # Enable pgvector extension
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

    # Create table
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS policy(
            id SERIAL PRIMARY KEY,
            content TEXT,
            embedding VECTOR(768)
        );
        """
    )

    conn.commit()
    cur.close()
    conn.close()
    print("Database is veified")

In [7]:
EMBEDDING_MODEL = "nomic-embed-text"

In [8]:
# Generate embedding
def generate_embedding(text_chunk: str)->list[float]:
    """ Call ollama instance to generate embedding """
    response = ollama.embeddings(model = EMBEDDING_MODEL, prompt = text_chunk)
    return response["embedding"]

In [9]:
# Store chunk into postgresql
def store_chunk_in_postgres(chunks: list[str]):
    """ Embed chunk and save/upsert them into postgreSQL with pgvector """
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    print("Embeding chunk and insert into PostgreSQL")
    for index, chunk in enumerate(chunks):
        if not chunk:
            continue

        # Generate 768 vector array via Ollama
        vector = generate_embedding(chunk)

        # Upsert query: Insert update existing chunk 
        cur.execute(
            """
            INSERT INTO policy(content, embedding)
            VALUES(%s, %s)        
            """, (chunk, vector)
        )
    conn.commit()
    cur.close()
    conn.close()
    print("Successfully ingest into PostgreSQL ...")

In [10]:
# 1. Initialize database table
setup_database()

# 2. Extract text from PDF and chunk it
PDF_FILE = "/home/mulyo/Projects/RAG/Corporate Policy Manual - Aether Dynamics.pdf"

# Generate safe-table chunk
safe_chunks = table_parsing(PDF_FILE)
print(f"Successfully generate {len(safe_chunks)} safe chunk")

# 3. Embed and save into pgvector
store_chunk_in_postgres(safe_chunks)

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Database is veified
Processing page 1 ...


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Processing page 2 ...


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Processing page 3 ...


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Processing page 4 ...
Processing page 5 ...
Processing page 6 ...
Processing page 7 ...
Processing page 8 ...
Processing page 9 ...
Processing page 10 ...
Successfully generate 90 safe chunk
Embeding chunk and insert into PostgreSQL
Successfully ingest into PostgreSQL ...
